**Imports & Authentications:**

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

In [ ]:
%load_ext google.colab.data_table

In [ ]:
import gspread
from google.auth import default
creds, _ = default()

# Authorize gspread
gc = gspread.authorize(creds)

In [ ]:
!pip install duckdb --quiet
import duckdb

**Querying Database (BigQuery):**

In [ ]:
%%bigquery dateframe_name --project arvig-report-data

# BigQuery Query Here

**Creating Google Sheet from dataframe:**

In [ ]:
from gspread_dataframe import set_with_dataframe

# 1. Create the new spreadsheet
sh = gc.create('dateframe_name')

# 2. Select the first worksheet
worksheet = sh.get_worksheet(0)

# 3. Write the dataframe (order_item_sample is the object created by your SQL cell)
set_with_dataframe(worksheet, dateframe_name)

sh.share('briean.truss.ra@arvig.com', perm_type='user', role='writer')

print(f"View sheet here: {sh.url}")

**Querying the dataframe with DuckDB:**

In [ ]:
# We use duckdb.query() to run SQL on the local DataFrame variable
query = """
SELECT
  Site,
  ReportCountyCode AS COUNTY,
  ReportCounty AS "CNTY DESCR",
  ReportCityCode AS "CTY TWP",
  ReportTownship AS "CTY TWP DESCR",
  ReportSortCode AS COLUMN,
  SUM(Quantity) AS "QTY MULT",
  SUM(Quantity) AS QTY,
  SUM(BilledAmount) AS "BILLED AMT"
FROM dateframe_name
GROUP BY Site, ReportSortCode, ReportCountyCode, ReportCounty, ReportTownship, ReportCityCode
ORDER BY Site, ReportSortCode, ReportCountyCode, ReportCounty, ReportTownship, ReportCityCode
"""

# Execute and convert back to a new DataFrame
catv_may_sub_counts_20260615 = duckdb.query(query).to_df()

# Display the first few rows
catv_may_sub_counts_20260615.head()